In [9]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
import pandas as pd

# 自定义数据集类，用于加载加利福尼亚房价数据
class CaliforniaHousingDataset(Dataset):
    def __init__(self, split='train', val_size=0.2, test_size=0.2, random_state=42):
        # 加载加利福尼亚房价数据集
        data = fetch_california_housing()
        X, y = data.data, data.target
        
        # 划分训练集、验证集和测试集
        from sklearn.model_selection import train_test_split
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=(val_size + test_size), random_state=random_state
        )
        val_relative_size = val_size / (val_size + test_size)
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_size/(val_size + test_size), random_state=random_state
        )
        
        # 使用训练集进行标准化
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        if split == 'train':
            self.X = torch.tensor(X_train, dtype=torch.float32)
            self.y = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        elif split == 'val':
            self.X = torch.tensor(X_val, dtype=torch.float32)
            self.y = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        elif split == 'test':
            self.X = torch.tensor(X_test, dtype=torch.float32)
            self.y = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
        else:
            raise ValueError("split must be 'train', 'val' or 'test'")
        
        self.n_samples = self.X.shape[0]
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 创建DataLoader函数
#batch_size: 批次大小，默认值为64
def get_california_housing_dataloader(batch_size=64, shuffle=True, num_workers=0):
    train_dataset = CaliforniaHousingDataset(split='train')
    val_dataset = CaliforniaHousingDataset(split='val')
    test_dataset = CaliforniaHousingDataset(split='test')
    
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers
    )
    
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )
    
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )
    
    return train_loader, val_loader, test_loader

#进行测试
train_loader, val_loader, test_loader = get_california_housing_dataloader()


In [10]:
# 查看一个批次的数据
for batch_X, batch_y in train_loader:
    print(batch_X.shape, batch_y.shape)
    break


torch.Size([64, 8]) torch.Size([64, 1])


In [11]:
import torch.nn as nn

class RegressionModel(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 30)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(30, 1)
    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

input_dim = next(iter(train_loader))[0].shape[1]
model = RegressionModel(input_dim)

In [12]:
model.eval()
with torch.no_grad():
    sample_X, sample_y = next(iter(train_loader))
    pred = model(sample_X)
    print("模型输出:", pred.shape)
    print("真实值:", sample_y.shape)

模型输出: torch.Size([64, 1])
真实值: torch.Size([64, 1])
